[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/TextureMapPrep_Colab.ipynb)  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/TextureMapPrep_Colab.ipynb)

# 🎨 TextureMapPrep — Seamless PBR Maps for Game Assets (Marigold + LOTUS + OpenCV + Real-ESRGAN)

**[Marigold](https://marigoldcomputervision.github.io/) + [LOTUS](https://lotus3d.github.io/) +
[OpenCV](https://opencv.org/) + [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)**
all wired into a single batch pipeline for converting one or many seamless **albedo**
PNG textures into the full **6-map PBR set** your game engine expects:

* **albedo** (base color, sRGB)
* **normal** (tangent-space, OpenGL convention — flip Y in Unity)
* **depth** (linear, 16-bit PNG)
* **height** (for displacement, derived from depth via Poisson)
* **roughness** (R channel, linear — from Marigold appearance IID)
* **metallic** (R channel, linear — from Marigold appearance IID)

…and then optionally upscales **every map** with Real-ESRGAN (2x or 4x) while
preserving the seamless wraparound.

The end-to-end pipeline:

```
albedo.png  ┌────────────────── MARIGOLD / LOTUS ────────────┐
            │  depth   →  height (Poisson)                    │
            │  normal  (tangent-space, OpenGL convention)     │
            │  albedo  (intrinsic decomposition)              │
            │  roughness / metallic (appearance IID)          │
            └──────────────────────────────────────────────┘
                ↓
            OpenCV classic fallbacks (normal-from-albedo via Sobel,
                                      normal-from-depth via cross product)
                ↓
            Real-ESRGAN 2x / 4x upscale (with tile-wrap seamless handling)
                ↓
6 PBR maps @ 1x or 2x/4x
```

## Why these models

* **[Marigold v1.1](https://huggingface.co/prs-eth/marigold-depth-v1-1)** (PRS-ETH,
  CVPR 2024 Oral, Best Paper Candidate) — Apache-2.0 code, RAIL++-M weights.
  One unified `diffusers` API for depth + surface normals + intrinsic image
  decomposition (albedo + roughness + metallic). Already in the diffusers core.
* **[LOTUS v1](https://lotus3d.github.io/)** (EnVision Research, ICLR 2025) —
  Apache-2.0 code + weights. The depth + normal model the
  [PBRFusion4DepthDemo-InstNorm](https://huggingface.co/spaces/NightRaven109/PBRFusion4DepthDemo-InstNorm)
  space uses. Comparable quality to Marigold, different architecture.
  Single-step inference = ~5x faster than standard diffusion-based methods.
* **[FLUX 2-klein-4B](https://huggingface.co/black-forest-labs/FLUX.2-klein-4B) +
  [NightRaven109/kleinalbedo4B5ksteps](https://huggingface.co/NightRaven109/kleinalbedo4B5ksteps)**
  (Apache-2.0) — the optional Albedo path from
  [PBRFusion4AlbedoKlein](https://huggingface.co/spaces/NightRaven109/PBRFusion4AlbedoKlein).
  Generates a clean, shadowless albedo from the input. ~8 GB. Off by default.
* **[OpenCV](https://opencv.org/)** (Apache-2.0) — classic normal-from-albedo
  (Sobel gradient), normal-from-depth (cross product on 3D points),
  height-from-normal (Poisson reconstruction via `skimage`), tile-wrap for
  seamless textures, all the basic PBR image ops.
* **[Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)** (BSD-3, 35.9k stars,
  on PyPI as `realesrgan`) — the standard 2x/4x texture super-resolution. Uses
  `tile` mode for large textures + `BORDER_WRAP` for seamless ones.

## How it differs from our other 3D/texture notebooks

| Notebook | Purpose |
|---|---|
| **Mesh_Optimizer** | post-process 3D meshes (decimate, repair, UV-unwrap, ground) |
| **SplatTransform** | 3DGS asset post-processor (SOG/SPLAT/SPZ) |
| **SkinTokens** | generate rigged characters from meshes |
| **Asset_Library_Browser** | browse, tag, and ship your 200+ assets |
| **TextureMapPrep (this)** | generate PBR map sets from albedo images |

This notebook slots in **before** a textured mesh enters `Mesh_Optimizer` /
`Pixal3D_Colab` / your game engine — it's the "I have an albedo, give me
the other 5 maps" stage.

## Pipeline diagram

```
seamless albedo.png  (one or many)
       ↓
[1] Marigold Depth       →  depth.png (16-bit)
[1] Marigold Normals     →  normal_map (used directly)
[1] Marigold IID         →  albedo_intrinsic, roughness, metallic
[1] (optional) FLUX Albedo  →  cleaner albedo_shadowless.png
       ↓
[2] OpenCV fallbacks     →  normal-from-albedo (if no normals)
                            normal-from-depth (verification)
                            height-from-normal (Poisson)
                            displacement.png
       ↓
[3] Real-ESRGAN 2x/4x    →  every map upscaled
                            (BORDER_WRAP to preserve seamless)
       ↓
/content/PBR_Out/<slug>/
  ├── albedo.png
  ├── albedo_shadowless.png  (if FLUX path)
  ├── normal.png
  ├── depth.png
  ├── height.png
  ├── displacement.png
  ├── roughness.png
  ├── metallic.png
  └── 2x/  or  4x/  (if upscaled)
```

## Requirements
* **GPU:** NVIDIA, ≥ 8 GB VRAM (T4 15 GB works for all paths except FLUX 2-klein
  which needs ~10 GB; FLUX is disabled by default on T4)
* **RAM:** ≥ 12 GB
* **Disk:** ≈ 5 GB free (PyTorch + diffusers + Marigold ~1.4 GB + LOTUS ~1.4 GB +
  Real-ESRGAN ~64 MB; FLUX 2-klein adds ~8 GB if enabled)
* **Time on first run:** 8-12 min (PyTorch + diffusers + all models)
* **Time on subsequent runs:** 1-2 min (everything cached in your Drive)
* **Per-texture runtime:** ~3-10 s for Marigold paths, ~20-40 s for FLUX path,
  ~1-2 s per upsample

## Where it fits in our pipeline
```
TextureMapPrep (this notebook)
   →  6 PBR maps per texture
   →  Mesh_Optimizer (if you want a quick preview mesh)
   →  Pixal3D_Colab / Hunyuan3D (full 3D from your PBR maps)
   →  Asset_Library_Browser
   →  Three.js / WebGPU game engine
```

> **⚠️ License note:** Marigold code is **Apache-2.0**, Marigold model weights are
> **RAIL++-M** (research + commercial w/ safety conditions). LOTUS is **Apache-2.0**
> (code + weights). FLUX 2-klein-4B is **Apache-2.0**. Real-ESRGAN is **BSD-3**.
> OpenCV is **Apache-2.0**. Most output maps are yours to use freely; the
> Marigold weights carry RAIL++-M conditions (mainly about safety classifiers).

In [ ]:
#@title STEP 1 — Install dependencies + clone repos
# Detects GPU + CUDA, installs torch 2.5.1+cu121, diffusers (with Marigold
# pipelines already in core), and clones the LOTUS repo. Real-ESRGAN installs
# from PyPI. FLUX 2-klein + Albedo LoRA are downloaded on first use in STEP 2
# to keep the install small (FLUX is optional).
import os, sys, time, json, subprocess, shutil, pathlib, traceback
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

print('='*72)
print('TextureMapPrep — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda or "none"})')
    print(f'  CUDA avail   : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f'  GPU {i}        : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected — most paths will be unusable on CPU')
except ImportError:
    torch = None
    print('  torch not yet installed (will be installed below)')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}  # info='Mount Google Drive and cache Marigold/LOTUS/Real-ESRGAN models + all generated PBR maps. Disable for ephemeral /content-only storage (models re-download each session).'
if CONNECT_GOOGLE_DRIVE:
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_Texture_Cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    print(f'  Drive cache  : {drive_root}')
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    os.environ['TORCH_HOME'] = str(drive_root / 'torch')
    os.environ['XDG_CACHE_HOME'] = str(drive_root / 'xdg')
    pbr_default = '/content/drive/MyDrive/AEI_PBR_Out'
else:
    drive_root = pathlib.Path('/content/_texture_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['TORCH_HOME'] = str(drive_root / 'torch')
    print(f'  Local cache  : {drive_root}  (lost on runtime disconnect)')
    pbr_default = '/content/PBR_Out'

REPO_DIR = drive_root / 'Lotus'
t_total = time.time()

# 1. PyTorch
if torch is None or not torch.cuda.is_available() or not torch.__version__.startswith('2.5'):
    print('\n[1/6] Installing PyTorch 2.5.1 + cu121 ...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
         'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
         '--index-url', 'https://download.pytorch.org/whl/cu121'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  ERROR:', r.stderr[-1500:])
        raise RuntimeError('PyTorch install failed')
    print(f'  PyTorch installed in {time.time()-t0:.0f}s')
else:
    print(f'\n[1/6] PyTorch {torch.__version__} already installed')

# 2. diffusers + Marigold + Real-ESRGAN
print('\n[2/6] Installing diffusers + Marigold + Real-ESRGAN ...')
t0 = time.time()
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     'diffusers>=0.30.0',
     'transformers>=4.41.0',
     'accelerate>=0.30.0',
     'safetensors', 'huggingface_hub',
     'opencv-python-headless==4.10.0.84',
     'pillow-heif', 'scipy', 'scikit-image', 'tqdm', 'matplotlib',
     'realesrgan',
     'color-matcher',
     'einops', 'omegaconf'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
print(f'  diffusers + Marigold + Real-ESRGAN installed in {time.time()-t0:.0f}s')

# 3. Clone LOTUS
if not (REPO_DIR / '.git').exists():
    print(f'\n[3/6] Cloning EnVision-Research/Lotus into {REPO_DIR} ...')
    t0 = time.time()
    r = subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/EnVision-Research/Lotus.git',
         str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  git clone failed:', r.stderr[-500:])
        raise RuntimeError('git clone failed')
    print(f'  cloned in {time.time()-t0:.0f}s')
else:
    print(f'\n[3/6] Reusing cached Lotus repo at {REPO_DIR}')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 4. Lotus deps
print('\n[4/6] Installing Lotus inference deps ...')
t0 = time.time()
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     'diffusers', 'transformers', 'accelerate',
     'scipy', 'numpy<2.0', 'opencv-python'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
print(f'  Lotus deps installed in {time.time()-t0:.0f}s')

# 5. Pre-download Marigold
print('\n[5/6] Pre-downloading Marigold model weights to Drive ...')
t0 = time.time()
from huggingface_hub import snapshot_download
marigold_cache = drive_root / 'marigold'
marigold_models = {
    'prs-eth/marigold-depth-v1-1':     ['unet', 'vae', 'scheduler', 'text_encoder', 'tokenizer', 'config.json'],
    'prs-eth/marigold-normals-v1-1':   ['unet', 'vae', 'scheduler', 'text_encoder', 'tokenizer', 'config.json'],
    'prs-eth/marigold-iid-appearance-v1-1': ['unet', 'vae', 'scheduler', 'text_encoder', 'tokenizer', 'config.json'],
    'prs-eth/marigold-iid-lighting-v1-1':   ['unet', 'vae', 'scheduler', 'text_encoder', 'tokenizer', 'config.json'],
}
for repo_id, patterns in marigold_models.items():
    sub = marigold_cache / repo_id.split('/')[-1]
    sub.mkdir(parents=True, exist_ok=True)
    try:
        snapshot_download(
            repo_id=repo_id,
            local_dir=str(sub),
            allow_patterns=patterns,
        )
        print(f'  cached : {repo_id}')
    except Exception as e:
        print(f'  WARN: {repo_id} download failed: {e}  (will retry on first use)')
print(f'  Marigold pre-download done in {time.time()-t0:.0f}s')

# 6. Pre-download LOTUS
print('\n[6/6] Pre-downloading LOTUS model weights to Drive ...')
t0 = time.time()
lotus_models = [
    'jingheya/lotus-depth-g-v2-1-disparity',
    'jingheya/lotus-depth-d-v2-0-disparity',
    'jingheya/lotus-normal-g-v1-1',
    'jingheya/lotus-normal-d-v1-1',
]
for repo_id in lotus_models:
    sub = drive_root / 'lotus' / repo_id.split('/')[-1]
    sub.mkdir(parents=True, exist_ok=True)
    try:
        snapshot_download(
            repo_id=repo_id,
            local_dir=str(sub),
        )
        print(f'  cached : {repo_id}')
    except Exception as e:
        print(f'  WARN: {repo_id} download failed: {e}  (will retry on first use)')
print(f'  LOTUS pre-download done in {time.time()-t0:.0f}s')

print()
print('='*72)
print(f'  STEP 1 complete  (total {time.time()-t_total:.0f}s)')
print('  Next: run STEP 2 (imports + lazy model cache)')
print('='*72)
print(f'  Default PBR output dir: {pbr_default}')


In [ ]:
#@title STEP 2 — Imports & lazy model cache
# Imports diffusers (Marigold pipelines), the LOTUS pipeline, OpenCV,
# scikit-image (Poisson), Real-ESRGAN, and the optional FLUX 2-klein Albedo
# LoRA path. All models are lazy-loaded on first use and cached so subsequent
# Gradio clicks are instant.
#
# Defines (all are top-level functions callable from STEPs 3-7).
#   - get_marigold(task)              -> MarigoldDepth/Normals/Intrinsics pipeline
#   - get_lotus(task, mode)           -> LotusGPipeline / LotusDPipeline
#   - get_flux_albedo()               -> FLUX 2-klein + Albedo LoRA (optional)
#   - get_realesrgan(scale)           -> RealESRGANer instance
#   - infer_depth_marigold(img, ...)  -> depth (Marigold)
#   - infer_normals_marigold(img, ...)-> tangent-space normals
#   - infer_iid_marigold(img, model)  -> {albedo, roughness, metallic} or {albedo, shading, residual}
#   - infer_depth_lotus(img, ...)     -> depth (LOTUS)
#   - infer_normals_lotus(img, ...)   -> normals (LOTUS)
#   - infer_albedo_flux(img, ...)     -> shadowless albedo (FLUX, optional)
#   - seamless_wrap(img)              -> wrap-then-crop helper
#   - normal_from_albedo(albedo)      -> OpenCV Sobel normal
#   - normal_from_depth(depth, K)     -> OpenCV cross-product normal
#   - height_from_normal(normal)      -> Poisson reconstruction
#   - displacement_from_height(h)     -> displacement map
import sys, os, time, json, pathlib, warnings, shutil
warnings.filterwarnings('ignore')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_DIR = pathlib.Path(os.environ.get('AEI_TEX_REPO',
                    str(pathlib.Path('/content/drive/MyDrive/AEI_Texture_Cache/Lotus'))))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f'  Lotus repo  : {REPO_DIR}')
print(f'  exists      : {REPO_DIR.exists()}')
print()

import torch
import numpy as np
print('  torch       :', torch.__version__)
print('  CUDA avail  :', torch.cuda.is_available())
print()

try:
    from diffusers import (
        MarigoldDepthPipeline,
        MarigoldNormalsPipeline,
        MarigoldIntrinsicsPipeline,
    )
    print('  diffusers.MarigoldDepthPipeline      OK')
    print('  diffusers.MarigoldNormalsPipeline    OK')
    print('  diffusers.MarigoldIntrinsicsPipeline OK')
except Exception as e:
    print(f'  diffusers import failed: {e}')

import cv2
print(f'  cv2          : {cv2.__version__}')

try:
    from skimage.restoration import inpaint_biharmonic
    print('  skimage.restoration.inpaint_biharmonic OK')
except Exception as e:
    print(f'  skimage not available (Poisson path disabled): {e}')

try:
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    print('  realesrgan.RealESRGANer OK')
except Exception as e:
    print(f'  realesrgan import failed: {e}')

from PIL import Image
print('  PIL.Image   OK')

MARIGOLD_DEPTH_REPO   = 'prs-eth/marigold-depth-v1-1'
MARIGOLD_NORMALS_REPO = 'prs-eth/marigold-normals-v1-1'
MARIGOLD_IID_APP_REPO = 'prs-eth/marigold-iid-appearance-v1-1'
MARIGOLD_IID_LIGHT_REPO = 'prs-eth/marigold-iid-lighting-v1-1'
MARIGOLD_CACHE = pathlib.Path('/content/drive/MyDrive/AEI_Texture_Cache/marigold')

LOTUS_DEPTH_G = 'jingheya/lotus-depth-g-v2-1-disparity'
LOTUS_DEPTH_D = 'jingheya/lotus-depth-d-v2-0-disparity'
LOTUS_NORMAL_G = 'jingheya/lotus-normal-g-v1-1'
LOTUS_NORMAL_D = 'jingheya/lotus-normal-d-v1-1'
LOTUS_CACHE = pathlib.Path('/content/drive/MyDrive/AEI_Texture_Cache/lotus')

FLUX_KLEIN_REPO = 'black-forest-labs/FLUX.2-klein-4B'
FLUX_ALBEDO_LORA = 'NightRaven109/kleinalbedo4B5ksteps'
FLUX_PROMPT = 'Basecolor, no shadows, flat lighting, keep color'

_PIPE_CACHE = {}
_LOTUS_CACHE = {}
_REALESRGAN_CACHE = {}
_FLUX_CACHE = {}

def get_marigold(task='depth', force_reload=False, fp16=True):
    # task in {'depth', 'normals', 'iid-appearance', 'iid-lighting'}
    key = ('marigold', task, fp16)
    if key in _PIPE_CACHE and not force_reload:
        return _PIPE_CACHE[key]
    repo = {
        'depth':           MARIGOLD_DEPTH_REPO,
        'normals':         MARIGOLD_NORMALS_REPO,
        'iid-appearance':  MARIGOLD_IID_APP_REPO,
        'iid-lighting':    MARIGOLD_IID_LIGHT_REPO,
    }[task]
    local = MARIGOLD_CACHE / repo.split('/')[-1]
    load_path = str(local) if (local / 'config.json').exists() else repo
    print(f'  Loading Marigold {task} from {load_path} ...')
    t0 = time.time()
    kwargs = {'variant': 'fp16', 'torch_dtype': torch.float16} if fp16 else {}
    if task == 'depth':
        pipe = MarigoldDepthPipeline.from_pretrained(load_path, **kwargs)
    elif task == 'normals':
        pipe = MarigoldNormalsPipeline.from_pretrained(load_path, **kwargs)
    elif task.startswith('iid'):
        pipe = MarigoldIntrinsicsPipeline.from_pretrained(load_path, **kwargs)
    else:
        raise ValueError(f'Unknown task: {task}')
    pipe = pipe.to('cuda')
    pipe.set_progress_bar_config(disable=True)
    _PIPE_CACHE[key] = pipe
    print(f'  Marigold {task} loaded in {time.time()-t0:.1f}s')
    return pipe

def get_lotus(task='depth', mode='generation', force_reload=False):
    # task in {'depth', 'normal'}, mode in {'generation', 'regression'}
    key = ('lotus', task, mode)
    if key in _LOTUS_CACHE and not force_reload:
        return _LOTUS_CACHE[key]
    if task == 'depth':
        repo = LOTUS_DEPTH_G if mode == 'generation' else LOTUS_DEPTH_D
    else:
        repo = LOTUS_NORMAL_G if mode == 'generation' else LOTUS_NORMAL_D
    local = LOTUS_CACHE / repo.split('/')[-1]
    load_path = str(local) if (local / 'model_index.json').exists() else repo
    print(f'  Loading LOTUS {task}/{mode} from {load_path} ...')
    t0 = time.time()
    from pipeline import LotusGPipeline, LotusDPipeline
    cls = LotusGPipeline if mode == 'generation' else LotusDPipeline
    pipe = cls.from_pretrained(load_path, torch_dtype=torch.float16).to('cuda')
    pipe.set_progress_bar_config(disable=True)
    _LOTUS_CACHE[key] = pipe
    print(f'  LOTUS {task}/{mode} loaded in {time.time()-t0:.1f}s')
    return pipe

def get_realesrgan(scale=4, force_reload=False):
    # Real-ESRGAN x4plus model. On first use, downloads to Drive (~64 MB).
    key = ('realesrgan', scale)
    if key in _REALESRGAN_CACHE and not force_reload:
        return _REALESRGAN_CACHE[key]
    cache_dir = pathlib.Path('/content/drive/MyDrive/AEI_Texture_Cache/realesrgan')
    cache_dir.mkdir(parents=True, exist_ok=True)
    model_path = cache_dir / 'RealESRGAN_x4plus.pth'
    if not model_path.exists():
        print(f'  Downloading RealESRGAN_x4plus.pth ...')
        import urllib.request
        urllib.request.urlretrieve(
            'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
            str(model_path))
    print(f'  Loading Real-ESRGAN (scale={scale}x) ...')
    t0 = time.time()
    rrdb = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                   num_block=23, num_grow_ch=32, scale=4)
    upsampler = RealESRGANer(
        scale=4, model_path=str(model_path), model=rrdb,
        tile=0, tile_pad=10, pre_pad=0, half=True,
    )
    _REALESRGAN_CACHE[key] = upsampler
    print(f'  Real-ESRGAN loaded in {time.time()-t0:.1f}s')
    return upsampler

def get_flux_albedo(force_reload=False):
    # FLUX 2-klein-4B + Albedo LoRA. Heavy: ~8 GB VRAM. Loads on demand.
    if 'flux' in _FLUX_CACHE and not force_reload:
        return _FLUX_CACHE['flux']
    print('  Loading FLUX 2-klein-4B + Albedo LoRA (~8 GB) ...')
    t0 = time.time()
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    from diffusers import Flux2KleinPipeline
    pipe = Flux2KleinPipeline.from_pretrained(
        FLUX_KLEIN_REPO, torch_dtype=torch.bfloat16,
    )
    lora_path = hf_hub_download(
        repo_id=FLUX_ALBEDO_LORA,
        filename='klein_albedo_5k_diffprompt.safetensors',
    )
    lora_sd = load_file(lora_path)
    model_sd = pipe.transformer.state_dict()
    SCALE = 1.0
    n_applied = 0
    for k in list(lora_sd.keys()):
        if not k.endswith('.lora_A.weight'):
            continue
        b_k = k.replace('.lora_A.weight', '.lora_B.weight')
        if b_k not in lora_sd:
            continue
        a = lora_sd[k].to(torch.bfloat16)
        b = lora_sd[b_k].to(torch.bfloat16)
        base = k.replace('.lora_A.weight', '').replace('diffusion_model.', '')
        # Find a matching key in the model
        target = None
        suffix = base.split('.')[-1]
        for mk in model_sd:
            if mk.endswith(suffix) and any(part in mk for part in base.split('.')):
                target = mk
                break
        if target and target in model_sd:
            try:
                delta = (b @ a)
                model_sd[target] = model_sd[target] + (delta * SCALE).to(model_sd[target].dtype)
                n_applied += 1
            except Exception:
                pass
    pipe.transformer.load_state_dict(model_sd)
    pipe = pipe.to('cuda')
    _FLUX_CACHE['flux'] = pipe
    print(f'  FLUX albedo loaded in {time.time()-t0:.1f}s  ({n_applied} LoRA keys applied)')
    return pipe

def load_image(path):
    img = Image.open(path)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    return np.array(img)

def save_image(arr, path):
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    if arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    Image.fromarray(arr).save(path)

def save_16bit(arr, path):
    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = np.clip(arr * 65535.0, 0, 65535).astype(np.uint16)
    Image.fromarray(arr, mode='I;16').save(path)

def seamless_wrap(img, border_px=64):
    # Wrap the image by `border_px` on each side so inference sees a seamless
    # texture, then crop the center back. Standard trick for making AI-generated
    # depth/normal outputs seamless across tile borders.
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    if border_px <= 0:
        return img
    return cv2.copyMakeBorder(
        img, border_px, border_px, border_px, border_px,
        borderType=cv2.BORDER_WRAP,
    )

def infer_depth_marigold(img, num_inference_steps=4, ensemble_size=1,
                          processing_resolution=768, seamless_border=64, seed=None):
    # Run Marigold depth. img = HxWx3 uint8. Returns HxW float32 [0,1].
    if seamless_border > 0:
        wrapped = seamless_wrap(img, seamless_border)
    else:
        wrapped = img
    pipe = get_marigold('depth')
    pil = Image.fromarray(wrapped)
    kwargs = dict(
        image=pil,
        num_inference_steps=int(num_inference_steps),
        ensemble_size=int(ensemble_size),
        processing_resolution=int(processing_resolution),
        match_input_resolution=True,
    )
    if seed is not None:
        kwargs['generator'] = torch.Generator(device='cuda').manual_seed(int(seed))
    out = pipe(**kwargs)
    depth = out.prediction[0]
    if seamless_border > 0:
        h, w = depth.shape
        depth = depth[seamless_border:h - seamless_border, seamless_border:w - seamless_border]
    return depth.astype(np.float32)

def infer_normals_marigold(img, num_inference_steps=4, ensemble_size=1,
                             processing_resolution=768, seamless_border=64, seed=None):
    # Run Marigold surface normals. Returns HxWx3 float32 in [-1, 1] (camera space).
    if seamless_border > 0:
        wrapped = seamless_wrap(img, seamless_border)
    else:
        wrapped = img
    pipe = get_marigold('normals')
    pil = Image.fromarray(wrapped)
    kwargs = dict(
        image=pil,
        num_inference_steps=int(num_inference_steps),
        ensemble_size=int(ensemble_size),
        processing_resolution=int(processing_resolution),
        match_input_resolution=True,
    )
    if seed is not None:
        kwargs['generator'] = torch.Generator(device='cuda').manual_seed(int(seed))
    out = pipe(**kwargs)
    n = out.prediction[0]
    if seamless_border > 0:
        h, w = n.shape[:2]
        n = n[seamless_border:h - seamless_border, seamless_border:w - seamless_border]
    return n.astype(np.float32)

def infer_iid_marigold(img, model='appearance', num_inference_steps=4, ensemble_size=1,
                        processing_resolution=768, seamless_border=64, seed=None):
    # Run Marigold IID. Returns dict with albedo (HxWx3), roughness (HxW), metallic (HxW)
    # for appearance; albedo + shading + residual for lighting.
    if seamless_border > 0:
        wrapped = seamless_wrap(img, seamless_border)
    else:
        wrapped = img
    task = 'iid-appearance' if model == 'appearance' else 'iid-lighting'
    pipe = get_marigold(task)
    pil = Image.fromarray(wrapped)
    kwargs = dict(
        image=pil,
        num_inference_steps=int(num_inference_steps),
        ensemble_size=int(ensemble_size),
        processing_resolution=int(processing_resolution),
        match_input_resolution=True,
    )
    if seed is not None:
        kwargs['generator'] = torch.Generator(device='cuda').manual_seed(int(seed))
    out = pipe(**kwargs)
    pred = out.prediction[0]
    result = {}
    sb = seamless_border
    for k, v in pred.items():
        if v.ndim == 3:
            v = v[sb:v.shape[0] - sb, sb:v.shape[1] - sb, :]
        else:
            v = v[sb:v.shape[0] - sb, sb:v.shape[1] - sb]
        result[k] = v.astype(np.float32)
    return result

def _lotus_preprocess(img, max_size=1024, min_size=384):
    # LOTUS preprocessing: resize to fit max 1024, min 384.
    # Returns fp16 tensor (1,3,H,W) on cuda and original (h, w).
    h, w = img.shape[:2]
    if max(h, w) > max_size:
        s = max_size / max(h, w)
    elif min(h, w) < min_size:
        s = min_size / min(h, w)
    else:
        s = 1.0
    nh, nw = int(h * s), int(w * s)
    img = cv2.resize(img, (nw, nh))
    arr = img.astype(np.float16)
    t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    t = t / 127.5 - 1.0
    return t.cuda().half(), (h, w)

def _lotus_postprocess_depth(pred, orig_hw):
    # LOTUS depth is mean of 3-channel output.
    d = pred.mean(axis=-1)
    d = cv2.resize(d, (orig_hw[1], orig_hw[0]))
    d_min, d_max = d.min(), d.max()
    d = (d - d_min) / (d_max - d_min + 1e-8)
    return d.astype(np.float32)

def _lotus_postprocess_normal(pred, orig_hw):
    # LOTUS normals are 3-channel [-1,1].
    n = cv2.resize(pred, (orig_hw[1], orig_hw[0]))
    n = n.astype(np.float32)
    n = n / (np.linalg.norm(n, axis=-1, keepdims=True) + 1e-8)
    return n

def _lotus_task_emb():
    e = torch.tensor([[1, 0]], dtype=torch.float32).cuda().half()
    return torch.cat([torch.sin(e), torch.cos(e)], dim=-1)

def infer_depth_lotus(img, mode='generation', seamless_border=64, seed=0):
    # Run LOTUS depth. Returns HxW float32 [0,1].
    if seamless_border > 0:
        wrapped = seamless_wrap(img, seamless_border)
    else:
        wrapped = img
    pipe = get_lotus('depth', mode=mode)
    t, orig_hw = _lotus_preprocess(wrapped)
    g = torch.Generator(device='cuda').manual_seed(int(seed)) if seed is not None else None
    with torch.inference_mode():
        out = pipe(rgb_in=t, prompt='', num_inference_steps=1, generator=g,
                   output_type='np', timesteps=[999], task_emb=_lotus_task_emb()).images[0]
    d = _lotus_postprocess_depth(out, orig_hw)
    if seamless_border > 0:
        h, w = d.shape
        d = d[seamless_border:h - seamless_border, seamless_border:w - seamless_border]
    return d.astype(np.float32)

def infer_normals_lotus(img, mode='generation', seamless_border=64, seed=0):
    # Run LOTUS normals. Returns HxWx3 float32 in [-1, 1].
    if seamless_border > 0:
        wrapped = seamless_wrap(img, seamless_border)
    else:
        wrapped = img
    pipe = get_lotus('normal', mode=mode)
    t, orig_hw = _lotus_preprocess(wrapped)
    g = torch.Generator(device='cuda').manual_seed(int(seed)) if seed is not None else None
    with torch.inference_mode():
        out = pipe(rgb_in=t, prompt='', num_inference_steps=1, generator=g,
                   output_type='np', timesteps=[999], task_emb=_lotus_task_emb()).images[0]
    n = _lotus_postprocess_normal(out, orig_hw)
    if seamless_border > 0:
        h, w = n.shape[:2]
        n = n[seamless_border:h - seamless_border, seamless_border:w - seamless_border]
    return n.astype(np.float32)

def infer_albedo_flux(img, width=1024, height=1024, num_inference_steps=1,
                       lora_weight=1.0, color_match_strength=1.0, seed=42):
    # Run FLUX 2-klein + Albedo LoRA + Reinhard color match. Optional path.
    # Returns HxWx3 uint8 (the shadowless albedo).
    pipe = get_flux_albedo()
    generator = torch.Generator(device='cuda').manual_seed(int(seed))
    pil = Image.fromarray(img).resize((width, height))
    out = pipe(
        prompt=FLUX_PROMPT,
        height=height, width=width,
        num_inference_steps=int(num_inference_steps),
        guidance_scale=1.0,
        generator=generator,
        image=[pil],
    ).images[0]
    if color_match_strength > 0:
        try:
            from color_matcher import ColorMatcher
            cm = ColorMatcher()
            target = np.array(out).astype(np.float32) / 255.0
            ref = np.array(pil).astype(np.float32) / 255.0
            matched = cm.transfer(src=target, ref=ref, method='reinhard')
            if color_match_strength < 1.0:
                matched = target + color_match_strength * (matched - target)
            matched = np.clip(matched * 255, 0, 255).astype(np.uint8)
            out = Image.fromarray(matched)
        except Exception as e:
            print(f'  WARN: Reinhard color match failed: {e}')
    return np.array(out).astype(np.uint8)

def normal_from_albedo(albedo, strength=2.0):
    # Derive a tangent-space normal map from the grayscale intensity of an
    # albedo via Sobel gradients. Returns HxWx3 uint8 (RGB).
    # Convention: (R, G, B) = (nx, ny, nz) in [0, 255]; nz=255 = facing camera.
    # OpenGL convention (Y up). Flip Y for DirectX/Unity.
    if albedo.ndim == 3:
        gray = cv2.cvtColor(albedo, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    else:
        gray = albedo.astype(np.float32) / 255.0
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3) * strength
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3) * strength
    n = np.stack([-gx, -gy, np.ones_like(gray)], axis=-1)
    n = n / (np.linalg.norm(n, axis=-1, keepdims=True) + 1e-8)
    n_enc = ((n * 0.5) + 0.5) * 255.0
    return np.clip(n_enc, 0, 255).astype(np.uint8)

def normal_from_depth(depth, fx=None, fy=None, cx=None, cy=None, strength=1.0):
    # Derive a camera-space normal from a depth map by computing the local
    # 3D point cloud and taking cross products. Returns HxWx3 float32 [-1,1].
    # If no intrinsics given, assumes 60-deg FOV.
    h, w = depth.shape
    if fx is None:
        f = max(h, w)
        fx = fy = f
        cx, cy = w / 2, h / 2
    z = depth.astype(np.float32)
    if z.max() > 0:
        z = z / z.max()
    z = z * 10.0 + 0.1
    us, vs = np.meshgrid(np.arange(w), np.arange(h))
    x = (us - cx) * z / fx
    y = (vs - cy) * z / fy
    pts = np.stack([x, y, z], axis=-1)
    pts_r = np.roll(pts, -1, axis=1)
    pts_d = np.roll(pts, -1, axis=0)
    n = np.cross(pts_r - pts, pts_d - pts) * strength
    n = n / (np.linalg.norm(n, axis=-1, keepdims=True) + 1e-8)
    return n.astype(np.float32)

def height_from_normal(normal, mask=None):
    # Reconstruct a height map from a surface normal map via Poisson
    # reconstruction (FFT-based). Returns HxW float32.
    # normal is HxWx3 in [-1, 1].
    nz = np.clip(normal[..., 2], 1e-3, 1.0)
    gx = -normal[..., 0] / nz
    gy = -normal[..., 1] / nz
    h, w = gx.shape
    gxx = np.fft.fft2(gx)
    gyy = np.fft.fft2(gy)
    fy = np.fft.fftfreq(h).reshape(-1, 1)
    fx = np.fft.fftfreq(w).reshape(1, -1)
    laplacian = -(2 * np.pi) ** 2 * (fx ** 2 + fy ** 2)
    laplacian[0, 0] = 1.0
    div = (1j * 2 * np.pi * fx) * gxx + (1j * 2 * np.pi * fy) * gyy
    h_freq = div / laplacian
    h_freq[0, 0] = 0.0
    height = np.real(np.fft.ifft2(h_freq))
    height = (height - height.min()) / (height.max() - height.min() + 1e-8)
    return height.astype(np.float32)

def displacement_from_height(height, scale=1.0):
    # Convert a [0,1] height map to a displacement map (centered, scaled).
    return ((height - 0.5) * 2.0 * float(scale) + 0.5).astype(np.float32)

def upscale_image(img, scale=4, seamless_border=32):
    # Upscale a single HxWx3 uint8 image by `scale`x using Real-ESRGAN.
    # For seamless textures, wraps first then crops to preserve tile continuity.
    if scale == 1:
        return img
    if seamless_border > 0:
        wrapped = seamless_wrap(img, seamless_border)
    else:
        wrapped = img
    upsampler = get_realesrgan(scale=scale)
    out, _ = upsampler.enhance(wrapped, outscale=scale)
    if seamless_border > 0:
        h, w = out.shape[:2]
        b = seamless_border * scale
        out = out[b:h - b, b:w - b]
    return out

print('  pipeline ready: Marigold + LOTUS + FLUX-albedo (opt) + Real-ESRGAN + OpenCV')
print('  seamless_wrap(), normal_from_albedo(), normal_from_depth(), height_from_normal()')


In [ ]:
#@title STEP 3 — Core helpers (single texture, batch, full PBR pipeline)
# Re-exports the inference functions from STEP 2 and adds
#   - run_full_pbr(image_path, output_dir, **kwargs) -> writes 6 PBR maps
#   - run_batch(input_dir, output_dir, **kwargs)     -> processes a folder
#   - normal_to_opengl/normal_to_directx             (engine conv helpers)
#   - make_seamless_check(img)                       (wrap-diff quality check)
import pathlib, json, traceback, time
import numpy as np
import cv2
from PIL import Image

def normal_to_opengl(normal):
    # Camera-space normal HxWx3 in [-1,1] (Y down) -> OpenGL tangent-space
    # normal map HxWx3 in [0, 255] (Y up).
    n = normal.copy()
    n[..., 1] = -n[..., 1]
    return ((n * 0.5) + 0.5) * 255.0

def normal_to_directx(normal):
    # Camera-space normal HxWx3 in [-1,1] (Y down) -> DirectX tangent-space
    # normal map HxWx3 in [0, 255] (Y down). No flip.
    return ((normal * 0.5) + 0.5) * 255.0

def make_seamless_check(img, axis='both'):
    # Wrap-shift the image by half on each axis, compute per-pixel diff.
    # Returns (diff_map, mean_diff). Lower = more seamless.
    h, w = img.shape[:2]
    if axis in ('horizontal', 'both'):
        a = np.roll(img, w // 2, axis=1)
        b = np.roll(a, -w // 2, axis=1)
    else:
        a, b = img, img
    if axis in ('vertical', 'both'):
        a2 = np.roll(a, h // 2, axis=0)
        b2 = np.roll(a2, -h // 2, axis=0)
    else:
        b2 = a
    diff = np.abs(b2.astype(np.float32) - a.astype(np.float32))
    if diff.ndim == 3:
        diff = diff.mean(axis=-1)
    return diff.astype(np.float32), float(diff.mean())

def run_full_pbr(image_path, output_dir,
                 depth_method='marigold',
                 depth_steps=4, depth_ensemble=1, depth_resolution=768,
                 normal_method='marigold',
                 normal_steps=4, normal_ensemble=1, normal_resolution=768,
                 normal_strength=2.0,
                 iid_model='appearance',
                 iid_steps=4, iid_ensemble=1, iid_resolution=768,
                 use_flux_albedo=False, flux_width=1024, flux_height=1024,
                 flux_steps=1, flux_lora_weight=1.0, flux_color_match=1.0,
                 use_height_from_normal=True, displacement_scale=1.0,
                 upscale_factor=1,
                 seamless_border=64,
                 normal_convention='opengl',
                 return_arrays=False,
                 progress_fn=None):
    # Process one seamless albedo into a full PBR set.
    img = np.array(Image.open(image_path).convert('RGB'))
    h0, w0 = img.shape[:2]
    out = pathlib.Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    written = {}

    def _progress(p, msg):
        if progress_fn:
            progress_fn(p, msg)
        print(f'  [{int(p*100):3d}%] {msg}')

    try:
        _progress(0.05, f'Running Marigold IID ({iid_model}) ...')
        iid = infer_iid_marigold(img, model=iid_model,
                                 num_inference_steps=iid_steps,
                                 ensemble_size=iid_ensemble,
                                 processing_resolution=iid_resolution,
                                 seamless_border=seamless_border)
        albedo_intrinsic = (np.clip(iid['albedo'], 0, 1) * 255.0).astype(np.uint8)
        if albedo_intrinsic.shape[:2] != (h0, w0):
            albedo_intrinsic = cv2.resize(albedo_intrinsic, (w0, h0))
        # Marigold IID albedo is in linear space; convert to sRGB for storage
        albedo_intrinsic_srgb = np.clip(
            np.power(albedo_intrinsic.astype(np.float32) / 255.0, 1.0 / 2.2) * 255.0, 0, 255
        ).astype(np.uint8)
        if iid_model == 'appearance':
            albedo_out = albedo_intrinsic_srgb
        else:
            albedo_out = img
        cv2.imwrite(str(out / 'albedo.png'),
                    cv2.cvtColor(albedo_out, cv2.COLOR_RGB2BGR))
        written['albedo'] = str(out / 'albedo.png')
        _progress(0.20, f'  -> albedo.png  {albedo_out.shape}')

        if use_flux_albedo:
            try:
                _progress(0.25, 'Running FLUX 2-klein Albedo path ...')
                albedo_flux = infer_albedo_flux(
                    img, width=flux_width, height=flux_height,
                    num_inference_steps=flux_steps,
                    lora_weight=flux_lora_weight,
                    color_match_strength=flux_color_match,
                )
                if albedo_flux.shape[:2] != (h0, w0):
                    albedo_flux = cv2.resize(albedo_flux, (w0, h0))
                cv2.imwrite(str(out / 'albedo_shadowless.png'),
                            cv2.cvtColor(albedo_flux, cv2.COLOR_RGB2BGR))
                written['albedo_shadowless'] = str(out / 'albedo_shadowless.png')
            except Exception as e:
                print(f'  WARN: FLUX albedo failed: {e}')

        if depth_method == 'marigold':
            _progress(0.35, 'Running Marigold depth ...')
            depth = infer_depth_marigold(img,
                                         num_inference_steps=depth_steps,
                                         ensemble_size=depth_ensemble,
                                         processing_resolution=depth_resolution,
                                         seamless_border=seamless_border)
        elif depth_method == 'lotus':
            _progress(0.35, 'Running LOTUS depth ...')
            depth = infer_depth_lotus(img, mode='generation',
                                      seamless_border=seamless_border)
        else:
            depth = None
        if depth is not None:
            if depth.shape != (h0, w0):
                depth = cv2.resize(depth, (w0, h0))
            save_16bit(depth, str(out / 'depth.png'))
            written['depth'] = str(out / 'depth.png')
            _progress(0.45, f'  -> depth.png  {depth.shape} 16-bit')

        if normal_method == 'marigold':
            _progress(0.55, 'Running Marigold normals ...')
            normal = infer_normals_marigold(img,
                                            num_inference_steps=normal_steps,
                                            ensemble_size=normal_ensemble,
                                            processing_resolution=normal_resolution,
                                            seamless_border=seamless_border)
        elif normal_method == 'lotus':
            _progress(0.55, 'Running LOTUS normals ...')
            normal = infer_normals_lotus(img, mode='generation',
                                         seamless_border=seamless_border)
        elif normal_method == 'opencv-sobel':
            _progress(0.55, 'Computing OpenCV normal-from-albedo ...')
            normal_float = normal_from_albedo(img, strength=normal_strength)
            normal = (normal_float.astype(np.float32) / 255.0) * 2.0 - 1.0
        elif normal_method == 'opencv-depth':
            if depth is None:
                _progress(0.55, 'No depth, falling back to OpenCV Sobel ...')
                normal_float = normal_from_albedo(img, strength=normal_strength)
                normal = (normal_float.astype(np.float32) / 255.0) * 2.0 - 1.0
            else:
                _progress(0.55, 'Computing OpenCV normal-from-depth ...')
                normal = normal_from_depth(depth, strength=normal_strength)
        else:
            normal = None
        if normal is not None:
            if normal.shape[:2] != (h0, w0):
                normal = cv2.resize(normal, (w0, h0))
            if normal_method in ('marigold', 'lotus', 'opencv-depth'):
                if normal_convention == 'opengl':
                    n_enc = normal_to_opengl(normal)
                else:
                    n_enc = normal_to_directx(normal)
            else:
                if normal_convention == 'opengl':
                    n_enc = normal_float.copy()
                    n_enc[..., 1] = 255 - n_enc[..., 1]
                else:
                    n_enc = normal_float
            cv2.imwrite(str(out / 'normal.png'),
                        cv2.cvtColor(n_enc, cv2.COLOR_RGB2BGR))
            written['normal'] = str(out / 'normal.png')
            _progress(0.65, f'  -> normal.png  {normal.shape} ({normal_convention})')

        if use_height_from_normal and normal is not None:
            _progress(0.72, 'Reconstructing height from normal (Poisson) ...')
            height = height_from_normal(normal)
            if height.shape != (h0, w0):
                height = cv2.resize(height, (w0, h0))
            save_16bit(height, str(out / 'height.png'))
            written['height'] = str(out / 'height.png')
            _progress(0.77, f'  -> height.png  {height.shape} 16-bit')
            disp = displacement_from_height(height, scale=displacement_scale)
            save_16bit(disp, str(out / 'displacement.png'))
            written['displacement'] = str(out / 'displacement.png')
            _progress(0.79, f'  -> displacement.png  {disp.shape} 16-bit')

        if iid_model == 'appearance' and 'roughness' in iid:
            rough = np.clip(iid['roughness'], 0, 1)
            if rough.shape != (h0, w0):
                rough = cv2.resize(rough, (w0, h0))
            save_16bit(rough, str(out / 'roughness.png'))
            written['roughness'] = str(out / 'roughness.png')
            _progress(0.82, f'  -> roughness.png  {rough.shape} 16-bit')
        if iid_model == 'appearance' and 'metallicity' in iid:
            metal = np.clip(iid['metallicity'], 0, 1)
            if metal.shape != (h0, w0):
                metal = cv2.resize(metal, (w0, h0))
            save_16bit(metal, str(out / 'metallic.png'))
            written['metallic'] = str(out / 'metallic.png')
            _progress(0.85, f'  -> metallic.png  {metal.shape} 16-bit')

        if upscale_factor > 1:
            _progress(0.90, f'Real-ESRGAN {upscale_factor}x upscale ...')
            up_dir = out / f'{upscale_factor}x'
            up_dir.mkdir(exist_ok=True)
            for k, p in written.items():
                if k in ('depth', 'height', 'displacement', 'roughness', 'metallic'):
                    continue
                arr = np.array(Image.open(p).convert('RGB'))
                up = upscale_image(arr, scale=upscale_factor, seamless_border=seamless_border)
                Image.fromarray(up).save(str(up_dir / pathlib.Path(p).name))
                written[f'{k}_{upscale_factor}x'] = str(up_dir / pathlib.Path(p).name)
            _progress(0.97, f'  -> {upscale_factor}x/  ({len(written)} files)')

        meta = {
            'source': str(image_path),
            'size': [h0, w0],
            'depth_method': depth_method,
            'normal_method': normal_method,
            'iid_model': iid_model,
            'use_flux_albedo': use_flux_albedo,
            'normal_convention': normal_convention,
            'upscale_factor': upscale_factor,
            'seamless_border': seamless_border,
            'files': written,
        }
        (out / 'meta.json').write_text(json.dumps(meta, indent=2))
        _progress(1.0, f'PBR set complete: {len(written)} files in {out}')
        if return_arrays:
            return out, written, {'albedo': albedo_out, 'depth': depth, 'normal': normal, 'iid': iid}
        return out, written, None
    except Exception as e:
        traceback.print_exc()
        raise

def run_batch(input_dir, output_dir, **kwargs):
    # Process every PNG/JPG in `input_dir` (or subdirs) as a separate texture.
    SUPP = {'.png', '.jpg', '.jpeg', '.webp'}
    in_dir = pathlib.Path(input_dir).resolve()
    if not in_dir.exists():
        raise SystemExit(f'Input not found: {in_dir}')
    out_dir = pathlib.Path(output_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    files = sorted(p for p in in_dir.rglob('*') if p.suffix.lower() in SUPP)
    if not files:
        raise SystemExit(f'No images found in {in_dir}')
    n_ok = 0
    n_skip = 0
    for src in files:
        slug = src.stem
        scene_out = out_dir / slug
        if (scene_out / 'meta.json').exists() and (scene_out / 'albedo.png').exists():
            print(f'  skip: {slug}  (already done)')
            n_skip += 1
            continue
        try:
            run_full_pbr(str(src), str(scene_out), **kwargs)
            n_ok += 1
        except Exception as e:
            print(f'  ERROR on {slug}: {type(e).__name__}: {e}')
    print(f'\n  done: {n_ok} new texture(s) processed, {n_skip} skipped')
    return n_ok, n_skip

print('  pipeline ready: run_full_pbr(), run_batch()')
print('  normal conventions: opengl (flip Y for Unity) / directx (no flip)')


In [ ]:
#@title STEP 4 — Launch Gradio UI
# Interactive Gradio UI for the full PBR pipeline.
# - Multi-image upload OR a folder of seamless textures
# - All major params exposed: depth/normal/IID method, steps, resolution,
#   FLUX albedo toggle, upscale factor, normal convention
# - 4 viewers: input preview, albedo, normal, depth
# - FileLink output for the meta.json + per-texture folder
import os, sys, time, json, shutil, pathlib, traceback, tempfile
import gradio as gr

gr.TEMP_DIR = '/tmp/gradio_texture'
os.makedirs(gr.TEMP_DIR, exist_ok=True)

def _do_run(files, input_folder,
            depth_method, depth_steps, depth_ensemble, depth_resolution,
            normal_method, normal_steps, normal_ensemble, normal_resolution,
            normal_strength,
            iid_model, iid_steps, iid_ensemble, iid_resolution,
            use_flux_albedo, flux_width, flux_height, flux_steps,
            flux_lora_weight, flux_color_match,
            use_height_from_normal, displacement_scale,
            upscale_factor, seamless_border, normal_convention,
            output_dir,
            progress=gr.Progress()):
    try:
        if not output_dir.strip():
            return 'Please set OUTPUT_DIR.', None, None, None, None, None
        out = pathlib.Path(output_dir).resolve()
        out.mkdir(parents=True, exist_ok=True)
        tmp = pathlib.Path(tempfile.mkdtemp(prefix='tex_gradio_'))
        # Determine input list
        image_paths = []
        if input_folder and pathlib.Path(input_folder).is_dir():
            SUPP = {'.png', '.jpg', '.jpeg', '.webp'}
            image_paths = sorted(p for p in pathlib.Path(input_folder).rglob('*')
                                  if p.suffix.lower() in SUPP)
        elif files:
            for f in files:
                src = pathlib.Path(f.name if hasattr(f, 'name') else f)
                dst = tmp / src.name
                shutil.copy2(str(src), str(dst))
                image_paths.append(dst)
        if not image_paths:
            return 'No images uploaded and no input folder set.', None, None, None, None, None
        # Process first image for preview
        first = image_paths[0]
        slug = first.stem
        progress(0.05, f'Processing {slug} (1 of {len(image_paths)}) ...')
        scene_out = out / slug
        scene_out.mkdir(parents=True, exist_ok=True)
        run_full_pbr(
            str(first), str(scene_out),
            depth_method=depth_method, depth_steps=depth_steps,
            depth_ensemble=depth_ensemble, depth_resolution=depth_resolution,
            normal_method=normal_method, normal_steps=normal_steps,
            normal_ensemble=normal_ensemble, normal_resolution=normal_resolution,
            normal_strength=float(normal_strength),
            iid_model=iid_model, iid_steps=iid_steps,
            iid_ensemble=iid_ensemble, iid_resolution=iid_resolution,
            use_flux_albedo=use_flux_albedo, flux_width=flux_width,
            flux_height=flux_height, flux_steps=flux_steps,
            flux_lora_weight=float(flux_lora_weight),
            flux_color_match=float(flux_color_match),
            use_height_from_normal=use_height_from_normal,
            displacement_scale=float(displacement_scale),
            upscale_factor=int(upscale_factor),
            seamless_border=int(seamless_border),
            normal_convention=normal_convention,
            progress_fn=lambda p, msg: progress(min(1.0, p), msg),
        )
        # Build preview images
        preview_albedo = str(scene_out / 'albedo.png')
        preview_normal = str(scene_out / 'normal.png')
        preview_depth = str(scene_out / 'depth.png')
        meta_path = str(scene_out / 'meta.json')
        # Process remaining images if batch
        if len(image_paths) > 1:
            progress(0.9, f'Processing {len(image_paths) - 1} more texture(s) ...')
            for p in image_paths[1:]:
                slug2 = p.stem
                scene_out2 = out / slug2
                if (scene_out2 / 'meta.json').exists():
                    print(f'  skip: {slug2}')
                    continue
                scene_out2.mkdir(parents=True, exist_ok=True)
                try:
                    run_full_pbr(
                        str(p), str(scene_out2),
                        depth_method=depth_method, depth_steps=depth_steps,
                        depth_ensemble=depth_ensemble, depth_resolution=depth_resolution,
                        normal_method=normal_method, normal_steps=normal_steps,
                        normal_ensemble=normal_ensemble, normal_resolution=normal_resolution,
                        normal_strength=float(normal_strength),
                        iid_model=iid_model, iid_steps=iid_steps,
                        iid_ensemble=iid_ensemble, iid_resolution=iid_resolution,
                        use_flux_albedo=use_flux_albedo, flux_width=flux_width,
                        flux_height=flux_height, flux_steps=flux_steps,
                        flux_lora_weight=float(flux_lora_weight),
                        flux_color_match=float(flux_color_match),
                        use_height_from_normal=use_height_from_normal,
                        displacement_scale=float(displacement_scale),
                        upscale_factor=int(upscale_factor),
                        seamless_border=int(seamless_border),
                        normal_convention=normal_convention,
                    )
                except Exception as e:
                    print(f'  ERROR on {slug2}: {e}')
        n = sum(1 for _ in out.rglob('meta.json'))
        return (f'Generated PBR set(s) for {slug} (and {n - 1} more in batch) -> '
                f'{out}'), preview_albedo, preview_normal, preview_depth, str(scene_out), meta_path
    except Exception as e:
        traceback.print_exc()
        return f'Error: {type(e).__name__}: {e}', None, None, None, None, None

with gr.Blocks(title='TextureMapPrep · PBR Maps (Marigold + LOTUS + Real-ESRGAN)') as demo:
    gr.Markdown(value=(
        '## 🎨 TextureMapPrep — Seamless PBR Maps\n'
        '[Marigold](https://marigoldcomputervision.github.io/) + [LOTUS](https://lotus3d.github.io/) + '
        'OpenCV + [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN) — convert seamless albedo PNGs '
        'into a full 6-map PBR set (albedo, normal, depth, height, roughness, metallic) with optional '
        '2x/4x upscale. Apache-2.0 + BSD-3 + OpenCV.'
    ))
    with gr.Row():
        with gr.Column(scale=1):
            files = gr.File(
                label='Albedo images  ( 1+ .png / .jpg, ideally seamless )',
                file_count='multiple',
                file_types=['.png', '.jpg', '.jpeg', '.webp'],
            )
            with gr.Accordion('Or pick a whole folder (batch mode)', open=False):
                input_folder = gr.Textbox(
                    label='Input folder  ( .png / .jpg / .webp recursively )',
                    value='', placeholder='/content/my_textures/',
                )
                output_dir = gr.Textbox(
                    label='Output folder  ( PBR sets per texture )',
                    value='/content/PBR_Out',
                )
            with gr.Accordion('Depth', open=False):
                depth_method = gr.Dropdown(
                    choices=['marigold', 'lotus', 'skip'], value='marigold',
                    label='Depth method',
                    info='Marigold = best quality (Apache-2.0 code, RAIL++-M model). LOTUS = single-step diffusion, also Apache-2.0. Skip = no depth (height-from-normal still works).'
                )
                depth_steps = gr.Slider(1, 50, value=4, step=1, label='Depth inference steps', info='More = slower + slightly sharper. Marigold default 4. 1 = fastest.')
                depth_ensemble = gr.Slider(1, 10, value=1, step=1, label='Depth ensemble size', info='Run inference N times with different latents and average. 1 = none. Higher = more accurate, Nx slower.')
                depth_resolution = gr.Slider(256, 1024, value=768, step=64, label='Depth processing resolution', info='Pad/crop input to this. Higher = more detail, more VRAM.')
            with gr.Accordion('Normals', open=False):
                normal_method = gr.Dropdown(
                    choices=['marigold', 'lotus', 'opencv-sobel', 'opencv-depth'], value='marigold',
                    label='Normal method',
                    info='Marigold = SOTA. LOTUS = alternative. opencv-sobel = classic gradient from albedo (lightweight). opencv-depth = cross product from depth.'
                )
                normal_steps = gr.Slider(1, 50, value=4, step=1, label='Normal inference steps', info='More = slightly sharper. Default 4.')
                normal_ensemble = gr.Slider(1, 10, value=1, step=1, label='Normal ensemble size', info='Average over N latents. Higher = more accurate.')
                normal_resolution = gr.Slider(256, 1024, value=768, step=64, label='Normal processing resolution', info='Higher = more detail.')
                normal_strength = gr.Slider(0.1, 10.0, value=2.0, step=0.1, label='Normal strength (OpenCV methods only)', info='Gradient multiplier. Higher = more pronounced normal bumps.')
                normal_convention = gr.Radio(
                    choices=['opengl', 'directx'], value='opengl',
                    label='Normal map convention',
                    info='OpenGL = Y up (Three.js, glTF 2.0). DirectX = Y down (Unity default). Flip Y in your engine if needed.'
                )
            with gr.Accordion('Intrinsic (Albedo + Roughness + Metallic)', open=False):
                iid_model = gr.Radio(
                    choices=['appearance', 'lighting'], value='appearance',
                    label='Marigold IID model',
                    info='Appearance = Albedo + Roughness + Metallic (PBR-ready). Lighting = Albedo + Shading + Residual (relighting). Use appearance for game assets.'
                )
                iid_steps = gr.Slider(1, 50, value=4, step=1, label='IID inference steps', info='More = slightly sharper. Default 4.')
                iid_ensemble = gr.Slider(1, 10, value=1, step=1, label='IID ensemble size', info='Average over N latents. Higher = more accurate.')
                iid_resolution = gr.Slider(256, 1024, value=768, step=64, label='IID processing resolution', info='Higher = more detail.')
            with gr.Accordion('FLUX Albedo (optional, heavy ~8 GB)', open=False):
                use_flux_albedo = gr.Checkbox(False, label='Use FLUX 2-klein + Albedo LoRA', info='Generate a clean shadowless albedo. ~8 GB VRAM, ~20-40s per texture. Off by default on T4.')
                flux_width = gr.Slider(256, 1024, value=1024, step=64, label='FLUX width', info='Output width. 1024 = full quality.')
                flux_height = gr.Slider(256, 1024, value=1024, step=64, label='FLUX height', info='Output height. 1024 = full quality.')
                flux_steps = gr.Slider(1, 4, value=1, step=1, label='FLUX inference steps', info='Distilled = 1 step is enough.')
                flux_lora_weight = gr.Slider(0.0, 2.0, value=1.0, step=0.05, label='FLUX LoRA weight', info='Strength of the Albedo LoRA. 1.0 = default.')
                flux_color_match = gr.Slider(0.0, 1.0, value=1.0, step=0.05, label='Reinhard color match strength', info='Transfer input color statistics to FLUX output. 1.0 = full.')
            with gr.Accordion('Height + Displacement', open=False):
                use_height_from_normal = gr.Checkbox(True, label='Reconstruct height from normal (Poisson)', info='FFT-based Poisson solver. Adds height.png + displacement.png.')
                displacement_scale = gr.Slider(0.1, 5.0, value=1.0, step=0.1, label='Displacement scale', info='Multiplier on the height-to-displacement conversion.')
            with gr.Accordion('Upscale (Real-ESRGAN, BSD-3)', open=False):
                upscale_factor = gr.Radio([1, 2, 4], value=1, label='Upscale factor', info='1 = no upscale. 2 = Real-ESRGAN 2x. 4 = Real-ESRGAN 4x. 16-bit maps (depth/height/etc) are not upscaled by default.')
                seamless_border = gr.Slider(0, 256, value=64, step=16, label='Seamless border (px)', info='Wrap-then-crop border for inference + upscale. Set to 0 if your input is NOT seamless.')
            run_btn = gr.Button('🚀 Run', variant='primary')
        with gr.Column(scale=1):
            log = gr.Textbox(label='Status', lines=2, interactive=False)
            with gr.Row():
                preview_albedo = gr.Image(label='Albedo (out)', type='filepath')
                preview_normal = gr.Image(label='Normal (out)', type='filepath')
            with gr.Row():
                preview_depth = gr.Image(label='Depth (out)', type='filepath')
                output_folder = gr.Textbox(label='Output folder', interactive=False)
            meta_out = gr.File(label='meta.json', interactive=False)
            gr.Markdown(value=(
                '**Next steps for the PBR set**\n'
                '* Pipe into **Mesh_Optimizer** for a quick preview mesh\n'
                '* Use in **Pixal3D_Colab** / **Hunyuan3D-2.1** for full 3D from your PBR maps\n'
                '* Pipe into **Asset_Library_Browser** to tag + ship\n'
                '* Load in Three.js / WebGPU / Babylon.js / PlayCanvas'
                '**Conventions**\n'
                '* Albedo = sRGB, 8-bit PNG\n'
                '* Normal = 8-bit PNG, RGB tangent-space (OpenGL by default)\n'
                '* Depth = 16-bit PNG, linear (1.0 = far)\n'
                '* Height = 16-bit PNG, [0, 1] (Poisson-reconstructed)\n'
                '* Roughness / Metallic = 16-bit PNG, R channel [0, 1]\n'
                '* Displacement = 16-bit PNG, [-scale, +scale]'
                '**For Unity**: flip the Green channel of normal.png (Y is inverted vs OpenGL).'
            ))
    run_btn.click(
        _do_run,
        inputs=[files, input_folder,
                depth_method, depth_steps, depth_ensemble, depth_resolution,
                normal_method, normal_steps, normal_ensemble, normal_resolution,
                normal_strength,
                iid_model, iid_steps, iid_ensemble, iid_resolution,
                use_flux_albedo, flux_width, flux_height, flux_steps,
                flux_lora_weight, flux_color_match,
                use_height_from_normal, displacement_scale,
                upscale_factor, seamless_border, normal_convention,
                output_dir],
        outputs=[log, preview_albedo, preview_normal, preview_depth, output_folder, meta_out],
    )
    def _welcome():
        return ('Ready. Upload 1+ seamless albedo PNGs (or set an input folder) '
                'and click Run. First image is previewed; remaining are batched.')
    demo.load(_welcome, outputs=[log])

from IPython.display import clear_output
clear_output()
demo.queue(default_concurrency_limit=2).launch(
    share=False, prevent_thread_lock=True, inline=True,
    show_error=True, height=1100,
)
print('\n  Gradio UI running above (cell stays alive - do not re-run)')


In [ ]:
#@title STEP 5 — Keep alive + session summary
import datetime, time, json, sys, pathlib, os, shutil
summary = {
    'timestamp'   : datetime.datetime.utcnow().isoformat() + 'Z',
    'python'      : sys.version.split()[0],
    'torch'       : None, 'cuda': None, 'gpus': [],
    'marigold_cache'   : str(MARIGOLD_CACHE),
    'lotus_cache'      : str(LOTUS_CACHE),
    'marigold_loaded'  : list({k[1] for k in _PIPE_CACHE.keys()}),
    'lotus_loaded'     : list({(k[1], k[2]) for k in _LOTUS_CACHE.keys()}),
    'flux_loaded'      : 'flux' in _FLUX_CACHE,
    'realesrgan_loaded': list({k[1] for k in _REALESRGAN_CACHE.keys()}),
    'ffmpeg'           : shutil.which('ffmpeg') is not None,
}
try:
    import torch
    summary['torch'] = torch.__version__
    summary['cuda']  = torch.version.cuda
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            summary['gpus'].append(f'{p.name}  {p.total_memory // (1024**3)} GB')
except Exception as e:
    summary['torch_error'] = str(e)
print(json.dumps(summary, indent=2))
print()
print('  License: Marigold (Apache-2.0 + RAIL++-M) + LOTUS (Apache-2.0) +')
print('           FLUX 2-klein (Apache-2.0) + Real-ESRGAN (BSD-3) + OpenCV (Apache-2.0)')
print()
print('  Cell will keep the runtime alive for 12 h unless you disconnect.')
print('  Press the stop button in the toolbar to release the GPU.')

_start = time.time()
while time.time() - _start < 43200:
    time.sleep(300)
    print(f'  [{int(time.time()-_start)//60} min] still running - close tab to stop')


In [ ]:
#@title STEP 6 — Quick test (Colab single-texture picker)
import time, pathlib

INPUT_PATH       = ''  #@param {type:'string', placeholder: '/content/my_albedo.png'}  # info='A single seamless albedo PNG. Optional: a folder path for batch mode.'
OUTPUT_DIR       = '/content/PBR_Out'  #@param {type:'string'}  # info='Where the per-texture PBR folder(s) will be written.'

DEPTH_METHOD     = 'marigold'  #@param ['marigold', 'lotus', 'skip']  # info='Marigold = best quality. LOTUS = alternative. Skip = no depth.'
DEPTH_STEPS      = 4  #@param {type:'slider', min:1, max:50, step:1}  # info='Depth inference steps. 4 = default.'
DEPTH_RESOLUTION = 768  #@param {type:'slider', min:256, max:1024, step:64}  # info='Depth processing resolution.'
NORMAL_METHOD    = 'marigold'  #@param ['marigold', 'lotus', 'opencv-sobel', 'opencv-depth']  # info='Marigold = SOTA. LOTUS = alt. opencv-sobel = lightweight. opencv-depth = cross product.'
NORMAL_STEPS     = 4  #@param {type:'slider', min:1, max:50, step:1}  # info='Normal inference steps. 4 = default.'
NORMAL_RESOLUTION = 768  #@param {type:'slider', min:256, max:1024, step:64}  # info='Normal processing resolution.'
NORMAL_STRENGTH  = 2.0  #@param {type:'slider', min:0.1, max:10, step:0.1}  # info='Gradient multiplier (OpenCV methods only).'
NORMAL_CONVENTION = 'opengl'  #@param ['opengl', 'directx']  # info='opengl = Y up (Three.js/glTF). directx = Y down (Unity).'
IID_MODEL        = 'appearance'  #@param ['appearance', 'lighting']  # info='Appearance = albedo+rough+metal. Lighting = albedo+shade+residual.'
USE_FLUX_ALBEDO  = False  #@param {type:'boolean'}  # info='Optional FLUX 2-klein Albedo path. ~8 GB VRAM, 20-40s per texture. Off by default.'
USE_HEIGHT       = True  #@param {type:'boolean'}  # info='Reconstruct height from normal via Poisson + write displacement.'
DISP_SCALE       = 1.0  #@param {type:'slider', min:0.1, max:5, step:0.1}  # info='Displacement map scale.'
UPSCALE_FACTOR   = 1  #@param [1, 2, 4]  # info='1=no upscale. 2 or 4 = Real-ESRGAN.'
SEAMLESS_BORDER  = 64  #@param {type:'slider', min:0, max:256, step:16}  # info='Wrap-then-crop border for inference + upscale. 0 if input is NOT seamless.'

if not INPUT_PATH.strip():
    raise SystemExit('Set INPUT_PATH to an albedo image or a folder.')
src = pathlib.Path(INPUT_PATH).resolve()
if not src.exists():
    raise SystemExit(f'Input not found: {src}')
out = pathlib.Path(OUTPUT_DIR).resolve()
out.mkdir(parents=True, exist_ok=True)
print(f'  input    : {src}')
print(f'  output   : {out}')
print(f'  depth    : {DEPTH_METHOD}  ({DEPTH_STEPS} steps, {DEPTH_RESOLUTION}px)')
print(f'  normal   : {NORMAL_METHOD}  ({NORMAL_STEPS} steps, {NORMAL_RESOLUTION}px, {NORMAL_CONVENTION})')
print(f'  IID      : {IID_MODEL}')
print(f'  FLUX     : {USE_FLUX_ALBEDO}')
print(f'  upscale  : {UPSCALE_FACTOR}x')
print()
t0 = time.time()
if src.is_dir():
    n_ok, n_skip = run_batch(
        str(src), str(out),
        depth_method=DEPTH_METHOD, depth_steps=DEPTH_STEPS, depth_resolution=DEPTH_RESOLUTION,
        normal_method=NORMAL_METHOD, normal_steps=NORMAL_STEPS, normal_resolution=NORMAL_RESOLUTION,
        normal_strength=NORMAL_STRENGTH, normal_convention=NORMAL_CONVENTION,
        iid_model=IID_MODEL,
        use_flux_albedo=USE_FLUX_ALBEDO,
        use_height_from_normal=USE_HEIGHT, displacement_scale=DISP_SCALE,
        upscale_factor=UPSCALE_FACTOR, seamless_border=SEAMLESS_BORDER,
    )
    print(f'\n  batch done: {n_ok} new texture(s), {n_skip} skipped, in {time.time()-t0:.0f}s')
else:
    slug = src.stem
    scene_out = out / slug
    scene_out.mkdir(parents=True, exist_ok=True)
    result = run_full_pbr(
        str(src), str(scene_out),
        depth_method=DEPTH_METHOD, depth_steps=DEPTH_STEPS, depth_resolution=DEPTH_RESOLUTION,
        normal_method=NORMAL_METHOD, normal_steps=NORMAL_STEPS, normal_resolution=NORMAL_RESOLUTION,
        normal_strength=NORMAL_STRENGTH, normal_convention=NORMAL_CONVENTION,
        iid_model=IID_MODEL,
        use_flux_albedo=USE_FLUX_ALBEDO,
        use_height_from_normal=USE_HEIGHT, displacement_scale=DISP_SCALE,
        upscale_factor=UPSCALE_FACTOR, seamless_border=SEAMLESS_BORDER,
    )
    if result is not None:
        out_dir, written, _ = result
        print(f'\n  done in {time.time()-t0:.0f}s')
        print(f'  output  : {out_dir}  ({len(written)} files)')
        for k, v in written.items():
            print(f'    {k}: {v}')


In [ ]:
#@title STEP 7 — Batch process a folder of textures
import time, pathlib

INPUT_DIR        = ''  #@param {type:'string', placeholder: '/content/my_textures/'}  # info='Folder containing PNG/JPG/WEBP albedo textures (recursive).'
OUTPUT_DIR       = '/content/PBR_Out'  #@param {type:'string'}  # info='Where per-texture PBR folders will be written.'

DEPTH_METHOD     = 'marigold'  #@param ['marigold', 'lotus', 'skip']  # info='Depth method for all textures in batch.'
DEPTH_STEPS      = 4  #@param {type:'slider', min:1, max:50, step:1}  # info='Depth inference steps.'
DEPTH_RESOLUTION = 768  #@param {type:'slider', min:256, max:1024, step:64}  # info='Depth processing resolution.'
NORMAL_METHOD    = 'marigold'  #@param ['marigold', 'lotus', 'opencv-sobel', 'opencv-depth']  # info='Normal method for all textures in batch.'
NORMAL_STEPS     = 4  #@param {type:'slider', min:1, max:50, step:1}  # info='Normal inference steps.'
NORMAL_RESOLUTION = 768  #@param {type:'slider', min:256, max:1024, step:64}  # info='Normal processing resolution.'
NORMAL_STRENGTH  = 2.0  #@param {type:'slider', min:0.1, max:10, step:0.1}  # info='Gradient multiplier (OpenCV methods only).'
NORMAL_CONVENTION = 'opengl'  #@param ['opengl', 'directx']  # info='opengl = Y up. directx = Y down.'
IID_MODEL        = 'appearance'  #@param ['appearance', 'lighting']  # info='Appearance = albedo+rough+metal. Lighting = albedo+shade+residual.'
USE_FLUX_ALBEDO  = False  #@param {type:'boolean'}  # info='Optional FLUX 2-klein Albedo path. ~8 GB VRAM. Off by default.'
USE_HEIGHT       = True  #@param {type:'boolean'}  # info='Reconstruct height from normal via Poisson.'
DISP_SCALE       = 1.0  #@param {type:'slider', min:0.1, max:5, step:0.1}  # info='Displacement map scale.'
UPSCALE_FACTOR   = 1  #@param [1, 2, 4]  # info='Real-ESRGAN 2x/4x per texture.'
SEAMLESS_BORDER  = 64  #@param {type:'slider', min:0, max:256, step:16}  # info='Wrap-then-crop border for inference + upscale.'

if not INPUT_DIR.strip():
    raise SystemExit('Set INPUT_DIR to a folder of albedo textures.')
in_dir = pathlib.Path(INPUT_DIR).resolve()
if not in_dir.exists():
    raise SystemExit(f'Input not found: {in_dir}')
out_dir = pathlib.Path(OUTPUT_DIR).resolve()
out_dir.mkdir(parents=True, exist_ok=True)
print(f'  input    : {in_dir}')
print(f'  output   : {out_dir}')
print(f'  depth    : {DEPTH_METHOD}  ({DEPTH_STEPS} steps, {DEPTH_RESOLUTION}px)')
print(f'  normal   : {NORMAL_METHOD}  ({NORMAL_STEPS} steps, {NORMAL_RESOLUTION}px, {NORMAL_CONVENTION})')
print(f'  IID      : {IID_MODEL}')
print(f'  FLUX     : {USE_FLUX_ALBEDO}')
print(f'  upscale  : {UPSCALE_FACTOR}x')
print()
t0 = time.time()
n_ok, n_skip = run_batch(
    str(in_dir), str(out_dir),
    depth_method=DEPTH_METHOD, depth_steps=DEPTH_STEPS, depth_resolution=DEPTH_RESOLUTION,
    normal_method=NORMAL_METHOD, normal_steps=NORMAL_STEPS, normal_resolution=NORMAL_RESOLUTION,
    normal_strength=NORMAL_STRENGTH, normal_convention=NORMAL_CONVENTION,
    iid_model=IID_MODEL,
    use_flux_albedo=USE_FLUX_ALBEDO,
    use_height_from_normal=USE_HEIGHT, displacement_scale=DISP_SCALE,
    upscale_factor=UPSCALE_FACTOR, seamless_border=SEAMLESS_BORDER,
)
print(f'\n  batch done: {n_ok} new texture(s), {n_skip} skipped, in {time.time()-t0:.0f}s')
